# Hypertension (CDC NHIS 2023) - 10 test records through the in-processing methods

Ten records of the untouched **test** split (fixed random seed). For every in-processing bias the table shows the ERM reference prediction and the prediction of the **selected** method (its probability, its decision at its own validation-chosen threshold, and whether that decision matches the recorded label), followed by the full method comparison used for the selection (validation split). The cross-hospital bias uses the cross-hospital ICU cohort instead: 5 stays from eICU test hospitals and 5 from MIMIC-IV (the external hospital), label = the recorded hypertension condition. Values come from the saved run (`hypertension/inprocessed/`); nothing is retrained here.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)
DATASET = "hypertension"
OUT = Path(r"D:/complete") / DATASET / "inprocessed"
sel = json.loads((OUT / "selection.json").read_text(encoding="utf-8"))
sample = pd.read_csv(OUT / "sample10_test_predictions.csv", low_memory=False)
hsample = pd.read_csv(OUT / "sample10_hospital_predictions.csv", low_memory=False)   # 5 eICU test-hospital + 5 MIMIC stays
ID_COL = "meta_patientunitstayid" if "meta_patientunitstayid" in sample else "meta_record_id"
GROUPS = ["sens_sex", "sens_age_group", "sens_race_ethnicity"]


def compare(method, hospital=False):
    frame = hsample if hospital else sample
    th = sel["thresholds_hospital"] if hospital else sel["thresholds"]
    show = (["meta_stay_id", "meta_source", "environment"] if hospital else [ID_COL]) + [c for c in GROUPS if c in frame] + ["target"]
    out = frame[show].copy()
    out["ERM probability"] = frame["p_ERM"].round(4)
    out["ERM decision"] = (frame["p_ERM"] >= th["ERM"]).astype(int)
    out[f"{method} probability"] = frame[f"p_{method}"].round(4)
    out[f"{method} decision"] = (frame[f"p_{method}"] >= th[method]).astype(int)
    out["ERM correct"] = out["ERM decision"] == out["target"]
    out[f"{method} correct"] = out[f"{method} decision"] == out["target"]
    return out.set_index(show[0])

## Demographic prediction dependence

In [ ]:
method = sel["selected"]["demographic"]
display(Markdown(f"**Selected:** {method} (methods compared: {', '.join(sel['families']['demographic']['methods'])})"))
display(compare(method, hospital=False))
display(pd.read_csv(OUT / "comparison_demographic.csv", index_col=0).round(4))

## Intersectional / subgroup unfairness

In [ ]:
method = sel["selected"]["intersectional"]
display(Markdown(f"**Selected:** {method} (methods compared: {', '.join(sel['families']['intersectional']['methods'])})"))
display(compare(method, hospital=False))
display(pd.read_csv(OUT / "comparison_intersectional.csv", index_col=0).round(4))

## Predictive miscalibration

In [ ]:
method = sel["selected"]["calibration"]
display(Markdown(f"**Selected:** {method} (methods compared: {', '.join(sel['families']['calibration']['methods'])})"))
display(compare(method, hospital=False))
display(pd.read_csv(OUT / "comparison_calibration.csv", index_col=0).round(4))

## Cross-hospital generalization failure

In [ ]:
method = sel["selected"]["environment"]
display(Markdown(f"**Selected:** {method} (methods compared: {', '.join(sel['families']['environment']['methods'])})"))
display(compare(method, hospital=True))
display(pd.read_csv(OUT / "comparison_environment.csv", index_col=0).round(4))